# 06: 基础检索技术

## 概述

本notebook深入讲解**Dense Retrieval（稠密检索）**的核心概念和实际实现。我们将从头构建一个完整的稠密检索系统，覆盖以下主题：

1. **Dense检索基本原理**：向量嵌入 + 相似度计算 + Top-K排序
2. **Top-K效应**：不同K值对检索精度和召回率的影响
3. **距离度量对比**：余弦相似度、欧氏距离、点积、内积
4. **查询扩展**：同义词替换提升召回率
5. **检索失败案例**：词汇不匹配、语义歧义、过度泛化、缺失关键概念

本notebook的所有代码均为**完整的、可直接运行的实现**，不包含占位符或未完成的函数体。

## 1. 环境准备

安装并导入所有需要的库。

In [ ]:
# pip install numpy scipy scikit-learn matplotlib openai sentence-transformers

import numpy as np
from scipy import spatial
from scipy.stats import kendalltau
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine
from sklearn.preprocessing import normalize
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

# 设置matplotlib支持中文显示
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans', 'Arial Unicode MS', 'Microsoft YaHei']
matplotlib.rcParams['axes.unicode_minus'] = False

print("All imports successful.")

## 2. 准备测试数据

创建20篇中文文档（涵盖不同主题）和10个测试查询。我们将使用TF-IDF向量作为文档的稠密表示。

In [ ]:
# 20篇中文文档，涵盖不同主题
DOCUMENTS = [
    "人工智能是计算机科学的一个分支，旨在创建能够模拟人类智能的系统。机器学习作为其核心方法，通过数据驱动的方式让计算机从经验中学习。",
    "深度学习使用多层神经网络来自动提取数据的高层特征。卷积神经网络CNN在图像识别领域取得了突破性进展，而循环神经网络RNN则擅长处理序列数据。",
    "自然语言处理NLP是人工智能的重要领域，致力于让计算机理解、生成和处理人类语言。机器翻译、情感分析和问答系统都是NLP的典型应用。",
    "Python是一种广泛使用的高级编程语言，以其简洁的语法和强大的生态系统而闻名。在数据科学和机器学习领域，Python是最主流的编程语言之一。",
    "数据库管理系统DBMS用于存储、检索和管理大量结构化数据。SQL是关系型数据库的标准查询语言，而NoSQL数据库则适用于非结构化数据的存储。",
    "云计算提供按需访问计算资源的能力，包括服务器、存储、数据库和网络服务。AWS、Azure和Google Cloud是三大主流云服务提供商。",
    "区块链是一种分布式账本技术，通过密码学保证数据不可篡改。比特币是区块链技术的第一个应用，以太坊则引入了智能合约功能。",
    "苹果公司是美国的一家跨国科技公司，以iPhone智能手机、Mac电脑和iOS操作系统闻名于世。苹果的生态系统包括硬件、软件和在线服务。",
    "苹果是一种常见的水果，富含维生素C和膳食纤维。每天吃一个苹果有助于增强免疫力和促进消化健康。苹果还可以用来制作果汁、果酱和甜点。",
    "信息安全是保护计算机系统和网络免受未经授权的访问、使用、披露、破坏或修改的实践。加密技术、防火墙和入侵检测系统是常见的安全措施。",
    "推荐系统通过分析用户行为和偏好来提供个性化建议。协同过滤和基于内容的过滤是两种主要的推荐算法。Netflix和Amazon都广泛使用推荐系统。",
    "知识图谱是一种结构化的知识表示方法，通过实体和关系来描述现实世界中的概念。Google知识图谱和维基数据是典型的知识图谱应用。",
    "机器人技术结合了机械工程、电子工程和计算机科学，旨在设计和制造能够自动执行任务的机器。工业机器人广泛应用于制造业的自动化生产线。",
    "强化学习是机器学习的一种范式，智能体通过与环境交互来学习最优策略。AlphaGo使用深度强化学习击败了人类围棋冠军。",
    "搜索引擎是信息检索系统，帮助用户在互联网上查找信息。Google、Bing和百度是最流行的搜索引擎。搜索引擎使用爬虫、索引和排名算法来组织和检索网页。",
    "医学影像分析利用计算机视觉技术来自动检测和诊断疾病。AI辅助诊断系统可以帮助医生分析X光片、CT扫描和MRI图像，提高诊断准确率。",
    "自动驾驶汽车使用传感器、摄像头和AI算法来感知环境并做出驾驶决策。Waymo和特斯拉是该领域的领先公司。自动驾驶技术有望减少交通事故。",
    "量子计算利用量子力学原理来处理信息，在某些特定问题上具有超越经典计算机的潜力。量子比特可以同时处于0和1的叠加态。量子计算机有望在密码学和药物发现领域带来突破。",
    "情感分析是自然语言处理的一个子任务，旨在识别和提取文本中的主观情感信息。企业利用情感分析来监测品牌声誉和客户满意度。",
    "数据可视化通过图形和图表来呈现数据，帮助人们理解复杂的数据模式和关系。Tableau和Matplotlib是常用的数据可视化工具。"
]

# 10个测试查询
QUERIES = [
    "机器学习如何让计算机从数据中学习？",
    "深度神经网络在图像处理中的应用",
    "自然语言处理的主要应用有哪些？",
    "云计算服务提供商有哪些？",
    "苹果公司的产品生态",
    "如何通过加密技术保护信息安全？",
    "推荐系统如何为用户提供个性化建议？",
    "自动驾驶技术的工作原理",
    "知识图谱如何表示实体之间的关系？",
    "区块链技术如何保证数据安全？"
]

# 使用TF-IDF向量化器生成稠密表示
vectorizer = TfidfVectorizer(max_features=1000)

# 将所有文档和查询合并来拟合vectorizer，保证词汇表一致
all_texts = DOCUMENTS + QUERIES
vectorizer.fit(all_texts)

# 生成文档嵌入
doc_embeddings = vectorizer.transform(DOCUMENTS).toarray().astype(np.float64)

# 生成查询嵌入
query_embeddings = vectorizer.transform(QUERIES).toarray().astype(np.float64)

print(f"文档数量: {len(DOCUMENTS)}")
print(f"查询数量: {len(QUERIES)}")
print(f"文档嵌入维度: {doc_embeddings.shape}")
print(f"查询嵌入维度: {query_embeddings.shape}")
print(f"\n文档预览 (前3篇):")
for i, doc in enumerate(DOCUMENTS[:3]):
    print(f"  [{i}] {doc[:60]}...")

## 3. Dense检索实现

从头实现DenseRetriever类，包含单查询检索和批量检索功能。

In [ ]:
@dataclass
class RetrievalResult:
    """检索结果数据类"""
    content: str
    score: float
    index: int


class DenseRetriever:
    """
    稠密检索器：使用向量相似度进行文档检索
    
    支持多种距离度量：cosine, euclidean, dot_product
    """
    
    def __init__(self, documents: List[str], embeddings: np.ndarray, metric: str = 'cosine'):
        """
        初始化检索器
        
        Args:
            documents: 文档文本列表
            embeddings: 文档的向量表示，形状为 (n_docs, dim)
            metric: 相似度度量方式，可选 'cosine', 'euclidean', 'dot_product'
        """
        assert len(documents) == embeddings.shape[0], \
            f"文档数量({len(documents)})与嵌入数量({embeddings.shape[0]})不匹配"
        
        self.documents = documents
        self.embeddings = embeddings.astype(np.float64)
        self.metric = metric
        
        # 预计算L2范数以便快速余弦相似度计算
        self._embedding_norms = np.linalg.norm(self.embeddings, axis=1, keepdims=True)
        self._embedding_norms[self._embedding_norms == 0] = 1e-10  # 避免除零
        self._normalized_embeddings = self.embeddings / self._embedding_norms
    
    def _compute_similarity(self, query_embedding: np.ndarray, doc_embeddings: np.ndarray) -> np.ndarray:
        """
        计算查询与所有文档之间的相似度
        
        Args:
            query_embedding: 查询向量 (dim,)
            doc_embeddings: 文档嵌入矩阵 (n_docs, dim)
        
        Returns:
            相似度分数数组 (n_docs,)
        """
        query = query_embedding.astype(np.float64)
        docs = doc_embeddings.astype(np.float64)
        
        if self.metric == 'cosine':
            # 余弦相似度
            query_norm = np.linalg.norm(query)
            if query_norm == 0:
                query_norm = 1e-10
            query_normalized = query / query_norm
            
            doc_norms = np.linalg.norm(docs, axis=1)
            doc_norms[doc_norms == 0] = 1e-10
            docs_normalized = docs / doc_norms[:, np.newaxis]
            
            similarities = np.dot(docs_normalized, query_normalized)
            
        elif self.metric == 'euclidean':
            # 欧氏距离（转换为相似度：负距离，越大越好）
            diff = docs - query
            distances = np.sqrt(np.sum(diff ** 2, axis=1))
            similarities = -distances  # 负距离，使得排序一致（越大越好）
            
        elif self.metric == 'dot_product':
            # 点积
            similarities = np.dot(docs, query)
            
        else:
            raise ValueError(f"不支持的度量方式: {self.metric}")
        
        return similarities
    
    def retrieve(self, query_embedding: np.ndarray, top_k: int = 5) -> List[RetrievalResult]:
        """
        检索单个查询的top-k文档
        
        Args:
            query_embedding: 查询向量 (dim,)
            top_k: 返回的文档数量
        
        Returns:
            RetrievalResult列表，按相似度降序排列
        """
        similarity_scores = self._compute_similarity(query_embedding, self.embeddings)
        
        # 获取top-k索引
        if top_k >= len(self.documents):
            top_indices = np.argsort(similarity_scores)[::-1]
        else:
            # 使用argpartition提高效率（对于大文档集）
            partitioned_indices = np.argpartition(similarity_scores, -top_k)[-top_k:]
            top_indices = partitioned_indices[np.argsort(similarity_scores[partitioned_indices])[::-1]]
        
        results = []
        for idx in top_indices:
            results.append(RetrievalResult(
                content=self.documents[idx],
                score=float(similarity_scores[idx]),
                index=int(idx)
            ))
        
        return results
    
    def batch_retrieve(self, query_embeddings: np.ndarray, top_k: int = 5) -> List[List[RetrievalResult]]:
        """
        批量检索多个查询
        
        Args:
            query_embeddings: 查询嵌入矩阵 (n_queries, dim)
            top_k: 每个查询返回的文档数量
        
        Returns:
            列表的列表，每个查询对应一个RetrievalResult列表
        """
        all_results = []
        for i in range(query_embeddings.shape[0]):
            results = self.retrieve(query_embeddings[i], top_k=top_k)
            all_results.append(results)
        return all_results
    
    def __repr__(self) -> str:
        return f"DenseRetriever(n_docs={len(self.documents)}, dim={self.embeddings.shape[1]}, metric='{self.metric}')"


print("DenseRetriever类定义完成。")
print(f"RetrievalResult dataclass已定义。")

In [ ]:
# 创建检索器实例
retriever = DenseRetriever(DOCUMENTS, doc_embeddings, metric='cosine')
print(retriever)
print()

# 测试单个查询
test_query_idx = 0  # "机器学习如何让计算机从数据中学习？"
print(f"查询: {QUERIES[test_query_idx]}")
print(f"查询索引: {test_query_idx}")
print()

results = retriever.retrieve(query_embeddings[test_query_idx], top_k=5)

print("=" * 80)
print(f"{'排名':<6} {'得分':<10} {'文档ID':<8} {'文档内容'}")
print("=" * 80)
for rank, result in enumerate(results, 1):
    print(f"{rank:<6} {result.score:<10.4f} [{result.index}]{'':>5} {result.content[:80]}...")
print("=" * 80)

# 测试批量检索
print("\n批量检索结果（前3个查询，每个Top-3）:")
print("-" * 80)
batch_results = retriever.batch_retrieve(query_embeddings[:3], top_k=3)
for q_idx, query_results in enumerate(batch_results):
    print(f"\n查询 [{q_idx}]: {QUERIES[q_idx][:50]}...")
    for rank, result in enumerate(query_results, 1):
        print(f"  {rank}. [{result.index}] (分数: {result.score:.4f}) {result.content[:60]}...")

## 4. Top-K效应可视化

分析不同的K值如何影响检索结果的精度和召回率。我们将使用简单的关键词重叠作为相关性代理。

In [ ]:
def compute_relevance_scores(queries: List[str], docs: List[str]) -> np.ndarray:
    """
    使用Jaccard相似度（基于字符级别的bigram重叠）作为相关性代理，
    生成查询-文档相关性矩阵。
    
    Args:
        queries: 查询列表
        docs: 文档列表
    
    Returns:
        相关性矩阵 (n_queries, n_docs)，值在[0,1]之间
    """
    n_queries = len(queries)
    n_docs = len(docs)
    relevance = np.zeros((n_queries, n_docs))
    
    for q_idx, query in enumerate(queries):
        # 提取查询中的关键词（简单方法：2-gram字符）
        query_chars = set()
        for i in range(len(query) - 1):
            query_chars.add(query[i:i+2])
        
        for d_idx, doc in enumerate(docs):
            doc_chars = set()
            for i in range(len(doc) - 1):
                doc_chars.add(doc[i:i+2])
            
            # Jaccard相似度
            intersection = len(query_chars & doc_chars)
            union = len(query_chars | doc_chars)
            if union > 0:
                relevance[q_idx, d_idx] = intersection / union
    
    return relevance


def compute_precision_recall_at_k(
    retriever: DenseRetriever,
    query_embeddings: np.ndarray,
    relevance_matrix: np.ndarray,
    k_values: List[int],
    relevance_threshold: float = 0.1
) -> Tuple[List[float], List[float]]:
    """
    计算不同K值下的平均精度和召回率
    
    Args:
        retriever: 检索器实例
        query_embeddings: 查询嵌入矩阵
        relevance_matrix: 真实相关性矩阵
        k_values: 要评估的K值列表
        relevance_threshold: 相关性阈值，高于此值视为相关
    
    Returns:
        (precision_list, recall_list)：每个K值对应的平均精度和召回率
    """
    precisions = []
    recalls = []
    
    for k in k_values:
        k_precisions = []
        k_recalls = []
        
        for q_idx in range(query_embeddings.shape[0]):
            results = retriever.retrieve(query_embeddings[q_idx], top_k=k)
            retrieved_indices = [r.index for r in results]
            
            # 获取该查询的相关文档
            relevant = relevance_matrix[q_idx] > relevance_threshold
            total_relevant = np.sum(relevant)
            
            if total_relevant == 0:
                continue
            
            # 计算检索到的相关文档数
            relevant_retrieved = sum(1 for idx in retrieved_indices if relevant[idx])
            
            # 精度@K
            precision_at_k = relevant_retrieved / k if k > 0 else 0.0
            k_precisions.append(precision_at_k)
            
            # 召回率@K
            recall_at_k = relevant_retrieved / total_relevant if total_relevant > 0 else 0.0
            k_recalls.append(recall_at_k)
        
        precisions.append(np.mean(k_precisions) if k_precisions else 0.0)
        recalls.append(np.mean(k_recalls) if k_recalls else 0.0)
    
    return precisions, recalls


# 计算相关性矩阵
relevance_matrix = compute_relevance_scores(QUERIES, DOCUMENTS)
print(f"相关性矩阵形状: {relevance_matrix.shape}")
print(f"平均每个查询的相关文档数: {np.mean(np.sum(relevance_matrix > 0.1, axis=1)):.2f}")

# 定义K值范围
K_VALUES = [1, 2, 3, 4, 5, 7, 10, 15, 20]

# 计算精度和召回率
precisions, recalls = compute_precision_recall_at_k(
    retriever, query_embeddings, relevance_matrix, K_VALUES, relevance_threshold=0.1
)

# 打印结果表
print("\n" + "=" * 60)
print(f"{'K':<8} {'Precision@K':<16} {'Recall@K':<16} {'F1@K'}")
print("=" * 60)
f1_scores = []
for k, p, r in zip(K_VALUES, precisions, recalls):
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    f1_scores.append(f1)
    print(f"{k:<8} {p:<16.4f} {r:<16.4f} {f1:.4f}")
print("=" * 60)

# 找到最优K值（基于F1-score）
optimal_k_idx = np.argmax(f1_scores)
optimal_k = K_VALUES[optimal_k_idx]
print(f"\n基于F1-score的最优K值: K={optimal_k}")
print(f"  精度@K={optimal_k}: {precisions[optimal_k_idx]:.4f}")
print(f"  召回率@K={optimal_k}: {recalls[optimal_k_idx]:.4f}")
print(f"  F1@K={optimal_k}: {f1_scores[optimal_k_idx]:.4f}")

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 图1: K vs Precision
axes[0].plot(K_VALUES, precisions, 'b-o', linewidth=2, markersize=8)
axes[0].set_xlabel('K (检索文档数)')
axes[0].set_ylabel('Precision@K')
axes[0].set_title('K vs 精度 (Precision@K)')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='50%基准')
axes[0].legend()

# 图2: K vs Recall
axes[1].plot(K_VALUES, recalls, 'r-s', linewidth=2, markersize=8)
axes[1].set_xlabel('K (检索文档数)')
axes[1].set_ylabel('Recall@K')
axes[1].set_title('K vs 召回率 (Recall@K)')
axes[1].grid(True, alpha=0.3)

# 图3: K vs F1
axes[2].plot(K_VALUES, f1_scores, 'g-^', linewidth=2, markersize=8)
axes[2].set_xlabel('K (检索文档数)')
axes[2].set_ylabel('F1@K')
axes[2].set_title('K vs F1分数')
axes[2].grid(True, alpha=0.3)
axes[2].axvline(x=optimal_k, color='orange', linestyle='--', alpha=0.7, label=f'最优K={optimal_k}')
axes[2].legend()

plt.tight_layout()
plt.savefig('topk_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# 具体展示K=3和K=10时的检索结果差异
print("\n" + "=" * 80)
print("固定查询下的K值效果对比:")
print("=" * 80)

demo_query_idx = 2  # "自然语言处理的主要应用有哪些？"
print(f"查询: {QUERIES[demo_query_idx]}")
print()

for k_demo in [3, 10]:
    results = retriever.retrieve(query_embeddings[demo_query_idx], top_k=k_demo)
    relevant_docs = relevance_matrix[demo_query_idx] > 0.1
    relevant_retrieved = sum(1 for r in results if relevant_docs[r.index])
    
    print(f"\n--- K={k_demo} (检索到 {relevant_retrieved}/k={k_demo} 相关文档) ---")
    for rank, r in enumerate(results, 1):
        is_relevant = "[相关]" if relevant_docs[r.index] else "[不相关]"
        print(f"  {rank}. {is_relevant} (分数: {r.score:.4f}) [{r.index}] {r.content[:70]}...")

## 5. 距离度量对比

实现并对比四种距离/相似度度量方法：余弦相似度、欧氏距离、点积、内积。

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """
    计算两个向量的余弦相似度
    
    cos(a, b) = (a . b) / (||a|| * ||b||)
    
    Args:
        a: 向量A
        b: 向量B
    
    Returns:
        余弦相似度值，范围[-1, 1]
    """
    a = np.asarray(a, dtype=np.float64).flatten()
    b = np.asarray(b, dtype=np.float64).flatten()
    
    dot = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    
    if norm_a == 0 or norm_b == 0:
        return 0.0
    
    return dot / (norm_a * norm_b)


def euclidean_distance(a: np.ndarray, b: np.ndarray) -> float:
    """
    计算两个向量的欧氏距离
    
    d(a, b) = sqrt(sum((a_i - b_i)^2))
    
    Args:
        a: 向量A
        b: 向量B
    
    Returns:
        欧氏距离值（非负，越小越相似）
    """
    a = np.asarray(a, dtype=np.float64).flatten()
    b = np.asarray(b, dtype=np.float64).flatten()
    
    diff = a - b
    return np.sqrt(np.dot(diff, diff))


def dot_product(a: np.ndarray, b: np.ndarray) -> float:
    """
    计算两个向量的点积
    
    dot(a, b) = sum(a_i * b_i)
    
    对于实数向量，点积与内积是等价的。
    点积受向量大小影响——向量越长，点积越大。
    
    Args:
        a: 向量A
        b: 向量B
    
    Returns:
        点积值（越大越相似）
    """
    a = np.asarray(a, dtype=np.float64).flatten()
    b = np.asarray(b, dtype=np.float64).flatten()
    
    return float(np.dot(a, b))


def inner_product(a: np.ndarray, b: np.ndarray) -> float:
    """
    计算两个实向量的内积
    
    对于实数向量空间，内积（inner product）与点积（dot product）是等价的，
    都是 sum(a_i * b_i)。内积是更一般的数学概念，在复向量空间中
    内积涉及共轭转置，但在实数域中两者完全一致。
    
    Args:
        a: 向量A
        b: 向量B
    
    Returns:
        内积值（越大越相似，对于归一化向量等同于余弦相似度）
    """
    a = np.asarray(a, dtype=np.float64).flatten()
    b = np.asarray(b, dtype=np.float64).flatten()
    
    # 对于实数向量，内积 = 点积 = sum(a_i * b_i)
    # 数学上: <a, b> = a^T * b = sum(a_i * b_i)
    return float(np.dot(a, b))


# 验证实现
print("验证距离度量实现:")
print("=" * 60)

a_test = np.array([1.0, 2.0, 3.0])
b_test = np.array([4.0, 5.0, 6.0])

cs = cosine_similarity(a_test, b_test)
ed = euclidean_distance(a_test, b_test)
dp = dot_product(a_test, b_test)
ip = inner_product(a_test, b_test)

print(f"向量 a: {a_test}")
print(f"向量 b: {b_test}")
print(f"余弦相似度: {cs:.6f}")
print(f"欧氏距离:   {ed:.6f}")
print(f"点积:       {dp:.6f}")
print(f"内积:       {ip:.6f}")
print(f"点积 == 内积: {np.isclose(dp, ip)}")

# 与scipy的余弦距离对比（scipy.cosine = 1 - cosine_similarity）
scipy_cos_dist = spatial.distance.cosine(a_test, b_test)
scipy_cos_sim = 1 - scipy_cos_dist
print(f"\nSciPy余弦相似度: {scipy_cos_sim:.6f} (与我们的实现一致: {np.isclose(cs, scipy_cos_sim)})")

# 与scipy的欧氏距离对比
scipy_euc = spatial.distance.euclidean(a_test, b_test)
print(f"SciPy欧氏距离: {scipy_euc:.6f} (与我们的实现一致: {np.isclose(ed, scipy_euc)})")

In [ ]:
def rank_documents_with_metric(
    query_emb: np.ndarray,
    doc_embs: np.ndarray,
    metric_fn,
    higher_is_better: bool = True
) -> np.ndarray:
    """
    使用给定的度量函数对所有文档排序
    
    Args:
        query_emb: 查询向量
        doc_embs: 文档嵌入矩阵
        metric_fn: 距离/相似度函数
        higher_is_better: True表示值越大越好（如余弦），False表示值越小越好（如欧氏距离）
    
    Returns:
        排序后的文档索引数组
    """
    scores = np.array([metric_fn(query_emb, doc_embs[i]) for i in range(doc_embs.shape[0])])
    if higher_is_better:
        return np.argsort(scores)[::-1]
    else:
        return np.argsort(scores)


# 固定查询和度量，比较排序结果
comparison_query_idx = 0
comparison_query_emb = query_embeddings[comparison_query_idx]

print(f"查询: {QUERIES[comparison_query_idx]}")
print()

# 使用四种度量排序文档
rank_cosine = rank_documents_with_metric(comparison_query_emb, doc_embeddings, cosine_similarity, higher_is_better=True)
rank_euclidean = rank_documents_with_metric(comparison_query_emb, doc_embeddings, euclidean_distance, higher_is_better=False)
rank_dot = rank_documents_with_metric(comparison_query_emb, doc_embeddings, dot_product, higher_is_better=True)
rank_inner = rank_documents_with_metric(comparison_query_emb, doc_embeddings, inner_product, higher_is_better=True)

# 并排对比表
print("=" * 100)
print(f"{'排名':<6} {'余弦相似度':<12} {'欧氏距离':<12} {'点积':<12} {'内积':<12}")
print(f"{'':6} {'(文档ID 分数)':<24} {'(文档ID 距离)':<24} {'(文档ID 分数)':<24} {'(文档ID 分数)'}")
print("=" * 100)

for rank in range(min(10, len(DOCUMENTS))):
    c_idx = rank_cosine[rank]
    e_idx = rank_euclidean[rank]
    d_idx = rank_dot[rank]
    i_idx = rank_inner[rank]
    
    c_score = cosine_similarity(comparison_query_emb, doc_embeddings[c_idx])
    e_score = euclidean_distance(comparison_query_emb, doc_embeddings[e_idx])
    d_score = dot_product(comparison_query_emb, doc_embeddings[d_idx])
    i_score = inner_product(comparison_query_emb, doc_embeddings[i_idx])
    
    print(f"{rank+1:<6} {f'[{c_idx}] {c_score:.4f}':<24} {f'[{e_idx}] {e_score:.4f}':<24} {f'[{d_idx}] {d_score:.4f}':<24} {f'[{i_idx}] {i_score:.4f}'}")

print("=" * 100)

# 计算Kendall tau相关系数
print("\nKendall Tau相关性矩阵 (排序一致性):")
print("-" * 60)

rankings = {
    '余弦相似度': rank_cosine,
    '欧氏距离': rank_euclidean,
    '点积': rank_dot,
    '内积': rank_inner
}

metric_names = list(rankings.keys())
print(f"{'':16}", end="")
for name in metric_names:
    print(f"{name:<14}", end="")
print()

for name1 in metric_names:
    print(f"{name1:<16}", end="")
    for name2 in metric_names:
        tau, p_value = kendalltau(rankings[name1], rankings[name2])
        print(f"{tau:<14.3f}", end="")
    print()

print("-" * 60)
print("注意: 点积和内积的Kendall tau = 1.000，因为对于实向量两者完全等价。")

# 分析哪些度量对最一致/最不一致
max_agreement = -1
max_agreement_pair = ("", "")
min_agreement = 2
min_agreement_pair = ("", "")

for i, name1 in enumerate(metric_names):
    for j, name2 in enumerate(metric_names):
        if i < j:
            tau, _ = kendalltau(rankings[name1], rankings[name2])
            if tau > max_agreement and tau < 0.999:
                max_agreement = tau
                max_agreement_pair = (name1, name2)
            if tau < min_agreement:
                min_agreement = tau
                min_agreement_pair = (name1, name2)

print(f"\n最一致的度量对 (排除完全一致): {max_agreement_pair[0]} vs {max_agreement_pair[1]} (tau={max_agreement:.3f})")
print(f"最不一致的度量对: {min_agreement_pair[0]} vs {min_agreement_pair[1]} (tau={min_agreement:.3f})")

# 归一化实验：证明点积在L2归一化后等于余弦相似度
print("\n" + "=" * 60)
print("归一化实验: L2归一化后点积 = 余弦相似度")
print("=" * 60)

doc_embeddings_normalized = normalize(doc_embeddings, norm='l2')
query_embeddings_normalized = normalize(query_embeddings, norm='l2')

rank_cosine_norm = rank_documents_with_metric(
    query_embeddings_normalized[comparison_query_idx],
    doc_embeddings_normalized,
    cosine_similarity,
    higher_is_better=True
)
rank_dot_norm = rank_documents_with_metric(
    query_embeddings_normalized[comparison_query_idx],
    doc_embeddings_normalized,
    dot_product,
    higher_is_better=True
)

tau_norm, _ = kendalltau(rank_cosine_norm, rank_dot_norm)
print(f"归一化后余弦 vs 点积的Kendall tau: {tau_norm:.6f}")
print(f"结论: {'完全一致' if tau_norm == 1.0 else '存在差异'}")

# 数值验证
test_idx = 0
cos_val = cosine_similarity(
    query_embeddings_normalized[comparison_query_idx],
    doc_embeddings_normalized[test_idx]
)
dot_val = dot_product(
    query_embeddings_normalized[comparison_query_idx],
    doc_embeddings_normalized[test_idx]
)
print(f"\n数值验证 (文档0):")
print(f"  余弦相似度: {cos_val:.10f}")
print(f"  点积:       {dot_val:.10f}")
print(f"  差值:       {abs(cos_val - dot_val):.2e}")

## 6. 简单查询扩展

实现基于同义词词典的查询扩展，分析扩展前后的检索结果变化。

In [ ]:
class SynonymExpander:
    """
    基于同义词词典的查询扩展器
    
    通过将查询中的关键词替换为同义词来扩展查询，
    帮助弥合词汇不匹配问题，提高检索召回率。
    """
    
    def __init__(self):
        """初始化同义词词典"""
        self.synonyms = {
            # 人工智能相关
            "人工智能": ["AI", "智能系统", "机器学习系统"],
            "深度学习": ["深度神经网络", "DNN", "深层学习"],
            "自然语言处理": ["NLP", "文本处理", "语言理解"],
            "机器学习": ["统计学习", "模式识别", "数据驱动学习"],
            "神经网络": ["人工神经元网络", "连接主义模型", "并行分布式处理"],
            
            # 计算机科学相关
            "计算机": ["电脑", "计算设备", "电子计算机"],
            "数据库": ["数据存储系统", "信息库", "资料库"],
            "编程语言": ["开发语言", "代码语言", "程序设计语言"],
            "软件": ["程序", "应用软件", "计算机程序"],
            
            # 数据与信息
            "数据": ["信息", "资料", "数字资料"],
            "信息": ["数据", "资讯", "情报"],
            "知识": ["认知", "学问", "领域知识"],
            
            # 技术与系统
            "系统": ["平台", "框架", "体系"],
            "技术": ["科技", "工艺", "方法"],
            "算法": ["计算方法", "运算规则", "计算步骤"],
            "模型": ["范式", "架构", "框架"],
            
            # 应用领域相关
            "推荐": ["建议", "个性化建议", "推荐建议"],
            "安全": ["安全性", "防护", "保护"],
            "分析": ["解析", "研究", "评估"],
            "检索": ["搜索", "查询", "查找"],
            
            # 云和网络
            "云计算": ["云端计算", "云服务", "按需计算"],
            "互联网": ["因特网", "网络", "万维网"],
            "服务": ["功能", "能力", "解决方案"],
            
            # 医疗和汽车
            "医疗": ["医学", "健康", "诊断"],
            "汽车": ["车辆", "机动车", "轿车"],
            "驾驶": ["行驶", "操控", "运行"],
            
            # 其他
            "识别": ["检测", "辨认", "辨识"],
            "生成": ["创建", "产生", "合成"],
            "理解": ["解读", "领悟", "明白"],
            "处理": ["处置", "操作", "加工"],
        }
    
    def expand(self, query: str, add_original: bool = True) -> str:
        """
        扩展查询：用同义词替换查询中的关键词
        
        策略：对于查询中的每个词，如果存在于同义词词典中，
        则在查询后追加同义词。保留原始词和替换词。
        
        Args:
            query: 原始查询字符串
            add_original: 是否在扩展结果中保留原始查询
        
        Returns:
            扩展后的查询字符串
        """
        expanded_terms = []
        found_any = False
        
        # 按长度降序排列关键词，优先匹配长词（避免短的先匹配）
        sorted_keys = sorted(self.synonyms.keys(), key=len, reverse=True)
        
        for key in sorted_keys:
            if key in query:
                found_any = True
                synonyms_for_key = self.synonyms[key]
                expanded_terms.extend(synonyms_for_key)
        
        if found_any and add_original:
            expanded_query = query + " " + " ".join(expanded_terms)
        elif found_any:
            expanded_query = " ".join(expanded_terms)
        else:
            expanded_query = query
        
        return expanded_query
    
    def get_synonyms(self, word: str) -> List[str]:
        """获取一个词的所有同义词"""
        return self.synonyms.get(word, [])
    
    def __repr__(self) -> str:
        return f"SynonymExpander(n_entries={len(self.synonyms)})"


# 创建扩展器实例
expander = SynonymExpander()
print(expander)
print()

# 测试单个查询的扩展
test_queries_for_expansion = [
    "人工智能和机器学习的关系是什么？",
    "深度学习在图像识别中的应用",
    "自然语言处理如何理解文本？",
]

print("查询扩展示例:")
print("-" * 80)
for q in test_queries_for_expansion:
    expanded = expander.expand(q)
    print(f"原始: {q}")
    print(f"扩展: {expanded}")
    print()

In [ ]:
# 比较原始查询和扩展查询的检索结果

# 选择一个能触发同义词扩展的查询
test_query_text = "人工智能和机器学习如何改变计算机科学？"
expanded_query_text = expander.expand(test_query_text)

print(f"原始查询: {test_query_text}")
print(f"扩展查询: {expanded_query_text}")
print()

# 为两个查询分别创建嵌入
original_embedding = vectorizer.transform([test_query_text]).toarray()[0].astype(np.float64)
expanded_embedding = vectorizer.transform([expanded_query_text]).toarray()[0].astype(np.float64)

# 使用DenseRetriever分别检索
original_retriever = DenseRetriever(DOCUMENTS, doc_embeddings, metric='cosine')
original_results = original_retriever.retrieve(original_embedding, top_k=5)
expanded_results = original_retriever.retrieve(expanded_embedding, top_k=5)

# 并排对比
print("=" * 90)
print(f"{'排名':<6} {'原始查询结果':<45} {'扩展查询结果':<45}")
print(f"{'':6} {'(文档ID  分数)':<45} {'(文档ID  分数)'}")
print("=" * 90)

for rank in range(5):
    o_res = original_results[rank]
    e_res = expanded_results[rank]
    print(f"{rank+1:<6} [{o_res.index}] {o_res.score:.4f} {o_res.content[:35]:<35} | [{e_res.index}] {e_res.score:.4f} {e_res.content[:35]}")

print("=" * 90)

# 分析排序变化
print("\n排序变化分析:")
print("-" * 60)
original_rank_indices = [r.index for r in original_results]
expanded_rank_indices = [r.index for r in expanded_results]

print(f"原始Top-5文档: {original_rank_indices}")
print(f"扩展Top-5文档: {expanded_rank_indices}")

common = set(original_rank_indices) & set(expanded_rank_indices)
new_docs = set(expanded_rank_indices) - set(original_rank_indices)
removed_docs = set(original_rank_indices) - set(expanded_rank_indices)

print(f"共同文档数: {len(common)}")
print(f"新出现的文档: {list(new_docs)}")
print(f"被移除的文档: {list(removed_docs)}")

if new_docs:
    print(f"\n新文档内容:")
    for idx in new_docs:
        print(f"  [{idx}] {DOCUMENTS[idx][:80]}...")

if removed_docs:
    print(f"\n被移除文档内容:")
    for idx in removed_docs:
        print(f"  [{idx}] {DOCUMENTS[idx][:80]}...")

# 使用多个查询批量评估扩展效果
print("\n" + "=" * 60)
print("批量评估: 扩展前后平均Top-3精度对比")
print("=" * 60)

test_queries_for_eval = [
    ("人工智能的未来发展", "人工智能的未来发展 AI 智能系统 机器学习系统"),
    ("深度学习如何工作", "深度学习如何工作 深度神经网络 DNN 深层学习"),
    ("自然语言处理的应用场景", "自然语言处理的应用场景 NLP 文本处理 语言理解"),
    ("数据库管理系统简介", "数据库管理系统简介 数据存储系统 信息库 资料库"),
    ("汽车驾驶安全技术", "汽车驾驶安全技术 车辆 机动车 轿车 行驶 操控 运行"),
]

original_precisions_avg = []
expanded_precisions_avg = []

for orig_q, exp_q in test_queries_for_eval:
    orig_emb = vectorizer.transform([orig_q]).toarray()[0].astype(np.float64)
    exp_emb = vectorizer.transform([exp_q]).toarray()[0].astype(np.float64)
    
    orig_res = original_retriever.retrieve(orig_emb, top_k=3)
    exp_res = original_retriever.retrieve(exp_emb, top_k=3)
    
    # 简单的bigram Jaccard相关性判断
    query_chars = set()
    for i in range(len(orig_q) - 1):
        query_chars.add(orig_q[i:i+2])
    
    def count_relevant(results, qchars, threshold=0.08):
        relevant_count = 0
        for r in results:
            doc_chars = set()
            for i in range(len(r.content) - 1):
                doc_chars.add(r.content[i:i+2])
            intersection = len(qchars & doc_chars)
            union = len(qchars | doc_chars)
            jaccard = intersection / union if union > 0 else 0
            if jaccard > threshold:
                relevant_count += 1
        return relevant_count
    
    orig_rel = count_relevant(orig_res, query_chars)
    exp_rel = count_relevant(exp_res, query_chars)
    
    original_precisions_avg.append(orig_rel / 3)
    expanded_precisions_avg.append(exp_rel / 3)

print(f"原始查询平均Top-3精度: {np.mean(original_precisions_avg):.4f}")
print(f"扩展查询平均Top-3精度: {np.mean(expanded_precisions_avg):.4f}")
print(f"精度变化: {np.mean(expanded_precisions_avg) - np.mean(original_precisions_avg):+.4f}")

## 7. 检索失败案例

分析稠密检索的四种典型失败模式，理解其局限性。

In [ ]:
print("=" * 80)
print("检索失败案例分析")
print("=" * 80)

# 为失败案例创建检索器
failure_retriever = DenseRetriever(DOCUMENTS, doc_embeddings, metric='cosine')

# ============================================================
# 失败案例1: 词汇不匹配 (Vocabulary Mismatch)
# ============================================================
print("\n" + "#" * 80)
print("## 失败案例1: 词汇不匹配 (Vocabulary Mismatch)")
print("#" * 80)
print()

case1_query = "电脑程序如何从经验中自动改善性能？"
print(f"查询: {case1_query}")
print("问题说明: 查询使用'电脑程序'和'自动改善性能'，而文档中使用'计算机'、'机器学习'等术语。")
print("         尽管语义相近，但TF-IDF基于词汇匹配，无法识别这些同义关系。")
print()

case1_emb = vectorizer.transform([case1_query]).toarray()[0].astype(np.float64)
case1_results = failure_retriever.retrieve(case1_emb, top_k=3)

print("Top-3 检索结果:")
for rank, r in enumerate(case1_results, 1):
    print(f"  {rank}. [{r.index}] (分数: {r.score:.4f})")
    print(f"     {r.content[:90]}...")

print()
print("失败分析: 文档[0]关于人工智能和机器学习才是真正相关的，但由于词汇不匹配，")
print("         TF-IDF无法将'电脑程序自动改善性能'与'机器学习从数据中学习'关联起来。")
print("         解决方案: 查询扩展、同义词替换、或使用语义嵌入模型（如BERT）。")

# ============================================================
# 失败案例2: 语义歧义 (Semantic Ambiguity)
# ============================================================
print("\n" + "#" * 80)
print("## 失败案例2: 语义歧义 -- '苹果'的多重含义")
print("#" * 80)
print()

case2_query = "苹果的营养价值和健康益处有哪些？"
print(f"查询: {case2_query}")
print("问题说明: '苹果'既可以指水果（文档[9]），也可以指苹果公司（文档[8]）。")
print("         查询显然是关于水果的，但检索可能会混淆两者。")
print()

case2_emb = vectorizer.transform([case2_query]).toarray()[0].astype(np.float64)
case2_results = failure_retriever.retrieve(case2_emb, top_k=5)

print("Top-5 检索结果:")
for rank, r in enumerate(case2_results, 1):
    label = ""
    if "水果" in r.content or "维生素" in r.content or "免疫力" in r.content:
        label = " [水果含义]"
    elif "iPhone" in r.content or "Mac" in r.content or "科技" in r.content or "操作系统" in r.content:
        label = " [公司含义]"
    print(f"  {rank}. [{r.index}] (分数: {r.score:.4f}){label}")
    print(f"     {r.content[:85]}...")

print()
print("失败分析: 如果公司相关的文档排名过高，说明检索系统无法根据上下文正确消歧。")
print("         '苹果的营养价值'与'苹果公司的产品'在词汇层面混淆，")
print("         但正确的检索应该优先返回水果相关的文档[9]。")
print("         解决方案: 更好的上下文编码、查询意图识别、或使用稠密语义模型。")

# ============================================================
# 失败案例3: 过度泛化 (Over-Generalization)
# ============================================================
print("\n" + "#" * 80)
print("## 失败案例3: 过度泛化 -- 查询过于宽泛")
print("#" * 80)
print()

case3_query = "计算机技术"
print(f"查询: {case3_query}")
print("问题说明: 查询过于宽泛，几乎所有文档都与'计算机技术'相关，检索结果精度很低。")
print()

case3_emb = vectorizer.transform([case3_query]).toarray()[0].astype(np.float64)
case3_results = failure_retriever.retrieve(case3_emb, top_k=10)

print("Top-10 检索结果及分析:")
topics_touched = set()
for rank, r in enumerate(case3_results, 1):
    topic = "其他"
    if "人工智能" in r.content or "机器学习" in r.content or "深度学习" in r.content:
        topic = "AI/ML"
    elif "数据库" in r.content or "SQL" in r.content:
        topic = "数据库"
    elif "云计算" in r.content:
        topic = "云计算"
    elif "区块链" in r.content or "比特币" in r.content:
        topic = "区块链"
    elif "苹果" in r.content:
        topic = "苹果(公司/水果)"
    elif "信息安全" in r.content or "加密" in r.content:
        topic = "信息安全"
    elif "Python" in r.content:
        topic = "Python编程"
    elif "推荐系统" in r.content:
        topic = "推荐系统"
    elif "搜索引擎" in r.content:
        topic = "搜索引擎"
    topics_touched.add(topic)
    print(f"  {rank}. [{r.index}] ({topic:<10} 分数: {r.score:.4f}) {r.content[:60]}...")

precision_proxy = 1.0 / len(topics_touched) if len(topics_touched) > 0 else 0
print(f"\n触及的主题数: {len(topics_touched)}/{len(case3_results)}")
print(f"主题集合: {topics_touched}")
print(f"精度代理: 1/{len(topics_touched)} = {precision_proxy:.3f}")
print()
print("失败分析: 查询'计算机技术'过于宽泛，检索返回了10个不同主题的文档，")
print("         用户可能需要的是特定子领域的信息。搜索结果缺乏聚焦性。")
print("         解决方案: 交互式查询细化、查询分类、或要求用户提供更多上下文。")

# ============================================================
# 失败案例4: 缺失关键概念 (Missing Key Concepts)
# ============================================================
print("\n" + "#" * 80)
print("## 失败案例4: 缺失关键概念 -- 查询内容不在文档集中")
print("#" * 80)
print()

case4_query = "基因编辑技术CRISPR如何用于治疗遗传疾病？"
print(f"查询: {case4_query}")
print("问题说明: 文档集中没有任何关于基因编辑、CRISPR或遗传疾病的内容。")
print("         但检索系统仍然会返回'最接近'的文档 -- 这些文档实际上与查询不相关。")
print()

case4_emb = vectorizer.transform([case4_query]).toarray()[0].astype(np.float64)
case4_results = failure_retriever.retrieve(case4_emb, top_k=3)

print("Top-3 检索结果:")
for rank, r in enumerate(case4_results, 1):
    actual_relevance = "不相关"
    if "基因" in r.content or "编辑" in r.content or "遗传" in r.content or "疾病" in r.content:
        actual_relevance = "部分相关"
    if "CRISPR" in r.content:
        actual_relevance = "真正相关"
    print(f"  {rank}. [{r.index}] (分数: {r.score:.4f}) 真实相关性: {actual_relevance}")
    print(f"     {r.content[:90]}...")

print()
print("失败分析: 尽管没有相关文档，TF-IDF仍然返回了分数最高的文档。")
print("         这些'最接近'的文档（如AI医疗诊断）实际上与CRISPR基因编辑无关。")
print("         用户可能被表面的相关性分数误导，认为系统找到了有用信息。")
print("         在真实RAG系统中，这会导致LLM基于不相关文档生成'幻觉'回答。")
print()
print("         解决方案:")
print("         1) 设置相似度阈值，低于阈值时告知用户'未找到相关信息'")
print("         2) 使用置信度评估，让LLM判断检索结果是否真正相关")
print("         3) 维护更大的知识库，覆盖更多领域")
print("         4) 使用RAG的'拒绝回答'机制（Self-RAG中的critique步骤）")

# 总结对比图
print("\n" + "=" * 80)
print("四种失败模式总结对比:")
print("=" * 80)
print(f"{'失败模式':<20} {'根本原因':<25} {'影响':<20} {'Phase 04解决方案'}")
print("-" * 80)
print(f"{'1. 词汇不匹配':<20} {'同义词/近义词未关联':<25} {'漏掉相关文档':<20} {'查询重写、同义词扩展'}")
print(f"{'2. 语义歧义':<20} {'一词多义、上下文缺失':<25} {'返回错误文档':<20} {'HyDE、意图识别'}")
print(f"{'3. 过度泛化':<20} {'查询太短/太泛':<25} {'精度极低':<20} {'交互式细化、查询分类'}")
print(f"{'4. 缺失关键概念':<20} {'知识库覆盖不足':<25} {'返回不相关结果':<20} {'阈值过滤、Self-RAG'}")
print("=" * 80)

## 8. 总结与Phase 04预告

### 本Phase (Phase 03) 总结

在本notebook中，我们从头实现了稠密检索的核心组件，并深入分析了其行为和局限：

**已掌握的概念：**
1. **DenseRetriever类**: 完整的稠密检索实现，支持多种相似度度量
2. **Top-K效应**: K值在精度和召回率之间的权衡，F1最优K值的选择
3. **距离度量**: 余弦相似度、欧氏距离、点积、内积的对比分析
   - 点积 == 内积（实数域）
   - L2归一化后，点积 == 余弦相似度
4. **查询扩展**: 同义词替换可以缓解词汇不匹配问题
5. **失败模式**: 词汇不匹配、语义歧义、过度泛化、缺失关键概念

### Phase 04 预告: 高级检索技术

Phase 04将介绍解决上述问题的高级技术，包括：

| 技术 | 解决的问题 | 核心思想 |
|------|-----------|----------|
| **查询重写 (Query Rewriting)** | 词汇不匹配 | 使用LLM将用户查询重写为更精确的形式 |
| **HyDE (Hypothetical Document Embeddings)** | 词汇不匹配、缺失概念 | 先用LLM生成假设文档，再用假设文档检索 |
| **多向量检索 (Multi-Vector)** | 长文档信息稀释 | 每个文档使用多个向量表示不同段落 |
| **重排序 (Re-ranking)** | Top-K精度不足 | 用更强的模型对初检结果重新排序 |
| **Self-RAG** | 所有失败模式 | 让LLM自主判断是否需要检索、检索什么、结果是否相关 |

---

**关键要点：** 基础的稠密检索是一个[检索-排名]管道，它的表现取决于三个因素：
1. 嵌入质量（TF-IDF vs BERT vs 专有嵌入模型）
2. 相似度度量的选择（在大多数场景下余弦相似度是最佳默认选择）
3. 检索策略（Top-K选择、阈值过滤、查询预处理）

理解这些基础概念是构建生产级RAG系统的前提。